## Richiesta 1

scrivi un codice python che legga tutti i dati da una tabella sql denominata "news_cyber" nello schema "cyber_risk" di un database postgresql denominato "text_analysis". Trasferisci tutti i dati in panda. Dopodiche esegui le seguenti operazioni:

1 - per ogni record considera il campo denominato "Body" ed applica sul testo contenuto in questo campo tutte le procedure standard di cleaning tipiche della text analysis utilizzando i tools della libreria nlpt. I testi sono in lingua inglese.

2 - salva il dataframe risultato dell'analisi in un nuovo dataframe

3- analizza il nuovo dataframe eliminando tutti i record uguali e tenendo traccia dei record eliminati in un secondo dataframe che chiamerai "log_analysis"

4 - salva entrambi i dataframe in formato csv

Cerca di rendere il codice il più robusto possibile, utilizza i costrutti try-catch  e cerca di chiarire sempre il tipo di errore che può essersi verificato . Commenta abbondamente il codice 

Di seguito un esempio di codice Python che svolge tutti i passaggi richiesti. Il codice è abbondantemente commentato, utilizza i costrutti `try-except` per gestire eventuali errori e cerca di essere il più robusto possibile. 

> **Nota**: 
> 1. Nel codice si fa uso di `psycopg2` per la connessione a PostgreSQL.  
> 2. Le “procedure standard di cleaning tipiche della text analysis” variano a seconda delle librerie e delle esigenze. In questo esempio, si fa riferimento alla rimozione di:  
>   - Punteggiatura  
>   - Caratteri speciali  
>   - Stopwords (lingua inglese)  
>   - Tokenizzazione e normalizzazione in minuscolo  
> 3. Viene ipotizzato che esista un modulo (o sottolibreria) denominato `nlpt` che contenga funzioni utili alla pulizia del testo. Se non esiste realmente, puoi sostituire le relative import/istruzioni con equivalenti della libreria `nltk` (o altra libreria di NLP).  
> 4. Ricordati di installare eventualmente le librerie necessarie (ad es. `pip install psycopg2 pandas nltk`).  

---

```python
import psycopg2
import pandas as pd
import logging

# Se si utilizzano funzioni di una libreria personalizzata 'nlpt', 
# sostituire questa import con le reali funzioni o con un'altra libreria
# specializzata in NLP (ad esempio 'nltk').
try:
    # Esempio (fittizio) di import dalla libreria 'nlpt'
    # from nlpt import clean_text
    # Se non esiste, si può usare NLTK, ad esempio:
    import nltk
    nltk.download('punkt')
    nltk.download('stopwords')
    from nltk.corpus import stopwords
    from nltk.tokenize import word_tokenize
    import string
except ImportError as e:
    logging.error("Errore nell'importare la libreria di NLP: %s", e)
    raise  # Rilancia l'eccezione per fermare l'esecuzione se non possiamo pulire il testo

# Configurazione di base per il logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Parametri di connessione al database (modificare in base alla propria configurazione)
DB_NAME = "text_analysis"
DB_USER = "postgres"
DB_PASSWORD = "password"
DB_HOST = "localhost"
DB_PORT = "5432"

# Funzione di pulizia testo (esempio generico di "standard cleaning")
def clean_text(text):
    """
    Esegue la pulizia di base di un testo in lingua inglese:
    1. Conversione in minuscolo
    2. Rimozione di punteggiatura
    3. Rimozione di caratteri speciali
    4. Tokenizzazione
    5. Rimozione stopwords
    6. Riassemblaggio del testo tokenizzato
    """
    if not isinstance(text, str):
        # Se 'text' non è una stringa, restituisce stringa vuota o un placeholder
        return ""

    # 1. Conversione in minuscolo
    text = text.lower()

    # 2. Rimozione della punteggiatura
    # (Qui consideriamo punteggiatura standard in 'string.punctuation')
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 3. Rimozione di eventuali caratteri speciali aggiuntivi (esempio)
    # Sostituiamo qualsiasi carattere non alfanumerico o spazio con uno spazio
    # se ritenuto necessario, altrimenti omettere questo passaggio.
    # import re
    import re
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # 4. Tokenizzazione
    tokens = word_tokenize(text)

    # 5. Rimozione stopwords (lingua inglese)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    # 6. Riassemblaggio del testo
    cleaned_text = " ".join(tokens)

    return cleaned_text


def main():
    """
    Funzione principale che:
    1. Legge tutti i dati dalla tabella 'news_cyber' dello schema 'cyber_risk'
       nel database 'text_analysis' (postgres).
    2. Trasferisce i dati in un dataframe pandas.
    3. Applica le procedure di cleaning sul campo 'Body'.
    4. Salva il dataframe risultato dell'analisi in un nuovo dataframe.
    5. Elimina i duplicati dal nuovo dataframe e tiene traccia dei record rimossi in 'log_analysis'.
    6. Salva entrambi i dataframe in formato CSV.
    """
    try:
        # 1. Connessione al database
        conn = psycopg2.connect(
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            host=DB_HOST,
            port=DB_PORT
        )
        logging.info("Connessione al database '%s' avvenuta con successo.", DB_NAME)

        # 2. Lettura dati dalla tabella 'news_cyber' (schema 'cyber_risk')
        query = """
            SELECT *
            FROM cyber_risk.news_cyber
        """
        df_original = pd.read_sql_query(query, con=conn)
        logging.info("Lettura dati completata. Numero di record: %d", len(df_original))

    except psycopg2.Error as e:
        logging.error("Errore nella connessione o esecuzione della query: %s", e)
        return  # Usciamo dal programma poiché non possiamo procedere senza dati
    finally:
        # Chiusura della connessione
        try:
            conn.close()
            logging.info("Connessione al database chiusa correttamente.")
        except Exception as e:
            logging.warning("Impossibile chiudere la connessione: %s", e)

    try:
        # 3. Applica le procedure di cleaning sul campo 'Body'
        #    Creo una copia del dataframe originale per sicurezza, se servisse
        df_processed = df_original.copy()

        # Applico la funzione di pulizia su ciascun record del campo 'Body'
        df_processed['Body'] = df_processed['Body'].apply(clean_text)

        # 4. Salva il dataframe risultato dell'analisi in un nuovo dataframe
        # (Per comodità, lo chiamiamo df_cleaned, ma è uguale a df_processed)
        df_cleaned = df_processed.copy()

        logging.info("Pulizia del testo completata.")

        # 5. Elimina i record duplicati e crea log dei record eliminati
        #    Troviamo i duplicati in base a tutte le colonne.
        duplicates_mask = df_cleaned.duplicated(keep='first')
        
        # Dataframe con i soli duplicati (quelli che verranno rimossi)
        log_analysis = df_cleaned[duplicates_mask].copy()

        # Ora rimuoviamo i duplicati da df_cleaned
        df_cleaned.drop_duplicates(keep='first', inplace=True)

        logging.info("Rimozione dei record duplicati completata. "
                     "Record totali rimanenti: %d, record duplicati: %d",
                     len(df_cleaned), len(log_analysis))

        # 6. Salva entrambi i dataframe in formato CSV
        #    (i nomi dei file .csv sono puramente indicativi)
        df_cleaned.to_csv("df_cleaned.csv", index=False)
        log_analysis.to_csv("log_analysis.csv", index=False)
        logging.info("Dataframe pulito e log di duplicati salvati in formato CSV.")

    except KeyError as e:
        logging.error("Errore nell'accesso a una colonna del dataframe: %s", e)
    except Exception as e:
        logging.error("Errore imprevisto nella pulizia dei testi: %s", e)


if __name__ == "__main__":
    main()
```

---

### Spiegazione passo-passo

1. **Import delle librerie**  
   - `psycopg2` per connettersi al database PostgreSQL.  
   - `pandas` per gestire i dataframe.  
   - `logging` per gestire i messaggi di log e gli errori.  
   - Le librerie NLP (ad esempio `nltk` o un ipotetico `nlpt`) per la pulizia del testo.

2. **Configurazione parametri di connessione**  
   - Variabili come `DB_NAME`, `DB_USER`, `DB_PASSWORD`, `DB_HOST` e `DB_PORT` devono essere aggiornate con i dati reali dell’ambiente di lavoro.

3. **Funzione `clean_text(text)`**  
   - Converte il testo in minuscolo.  
   - Rimuove la punteggiatura e i caratteri speciali.  
   - Esegue la tokenizzazione.  
   - Rimuove le stopwords (in lingua inglese).  
   - Riassembla il testo pulito.

4. **Funzione `main()`**  
   - Si connette al database PostgreSQL utilizzando `psycopg2.connect()` all’interno di un blocco `try-except` per catturare eventuali errori di connessione.  
   - Esegue la query SQL per selezionare tutti i record dalla tabella `news_cyber` nello schema `cyber_risk`.  
   - Trasferisce i dati letti in un dataframe `df_original`.  
   - Chiude la connessione all’interno di un blocco `finally` per garantire che venga rilasciata correttamente anche se si verifica un errore.  
   - Copia il dataframe originale in `df_processed` e applica la funzione di pulizia `clean_text` sul campo `"Body"`.  
   - Copia il dataframe processato in `df_cleaned`.  
   - Individua i record duplicati e crea un dataframe `log_analysis` con tali record.  
   - Elimina i duplicati da `df_cleaned`.  
   - Salva `df_cleaned` e `log_analysis` in due file CSV separati.

5. **Blocco `if __name__ == "__main__":`**  
   - Consente di eseguire la funzione `main()` solo quando lo script viene eseguito direttamente, non quando è importato come modulo.

6. **Gestione degli errori**  
   - `except psycopg2.Error as e:` per gli errori di connessione o di query.  
   - `except KeyError as e:` per errori nell’accesso a colonne inesistenti nel dataframe.  
   - `except Exception as e:` per qualsiasi altro errore imprevisto.  

In questo modo il codice risulta modulare, estensibile e abbastanza “robusto” per gestire gli errori più comuni che potrebbero verificarsi durante la connessione al database, la lettura dei dati, e la pulizia del testo.

## Richiesta 2

nel cleaning aggiungi anche la rimozione di tutti i numeri presenti nel testo

Di seguito troverai un esempio di codice Python che:

1. Si connette a un database PostgreSQL denominato **"text_analysis"** e legge tutti i dati dalla tabella **"news_cyber"** nello schema **"cyber_risk"**.  
2. Trasferisce i dati in un DataFrame di **pandas**.  
3. Applica le procedure di cleaning sul campo **"Body"**, tra cui:  
   - Conversione in minuscolo  
   - Rimozione di punteggiatura  
   - Rimozione di numeri  
   - Rimozione di caratteri speciali  
   - Tokenizzazione  
   - Rimozione di stopwords in lingua inglese  
   - Riassemblaggio del testo  
4. Salva il risultato della pulizia in un nuovo DataFrame.  
5. Elimina i duplicati dal nuovo DataFrame, tenendo traccia dei record rimossi in un secondo DataFrame denominato **"log_analysis"**.  
6. Salva entrambi i DataFrame in formato CSV.  

Il codice è scritto in modo “robusto” con blocchi `try-except` per gestire potenziali errori in modo appropriato.  

---

```python
import psycopg2
import pandas as pd
import logging

# Se utilizzi una libreria di NLP personalizzata (es. 'nlpt'), sostituisci le import di nltk
try:
    import nltk
    nltk.download('punkt')
    nltk.download('stopwords')
    from nltk.corpus import stopwords
    from nltk.tokenize import word_tokenize
    import string
except ImportError as e:
    logging.error("Errore nell'importare la libreria di NLP: %s", e)
    raise  # Interrompiamo l'esecuzione se non possiamo effettuare il cleaning del testo

# Configurazione di base per il logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Parametri di connessione al database (modificare in base alla propria configurazione)
DB_NAME = "text_analysis"
DB_USER = "postgres"
DB_PASSWORD = "password"
DB_HOST = "localhost"
DB_PORT = "5432"

def clean_text(text: str) -> str:
    """
    Esegue la pulizia di base di un testo in lingua inglese:
    1. Conversione in minuscolo
    2. Rimozione della punteggiatura
    3. Rimozione dei numeri
    4. Rimozione di eventuali caratteri speciali
    5. Tokenizzazione
    6. Rimozione stopwords (lingua inglese)
    7. Riassemblaggio del testo

    Restituisce la stringa pulita.
    """
    if not isinstance(text, str):
        # Se 'text' non è una stringa, restituisce stringa vuota
        return ""

    # 1. Conversione in minuscolo
    text = text.lower()

    # 2. Rimozione della punteggiatura
    #    (Qui consideriamo la punteggiatura definita in 'string.punctuation')
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 3. Rimozione dei numeri
    import re
    text = re.sub(r'\d+', '', text)

    # 4. Rimozione di eventuali caratteri speciali aggiuntivi (se necessario)
    #    Per esempio, sostituiamo qualsiasi carattere non alfanumerico o spazio con uno spazio.
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # 5. Tokenizzazione
    tokens = word_tokenize(text)

    # 6. Rimozione stopwords (lingua inglese)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    # 7. Riassemblaggio del testo
    cleaned_text = " ".join(tokens)
    
    return cleaned_text

def main():
    """
    Funzione principale che:
    1. Legge tutti i dati dalla tabella 'news_cyber' dello schema 'cyber_risk'
       nel database 'text_analysis'.
    2. Trasferisce i dati in un dataframe pandas.
    3. Applica il cleaning sul campo 'Body'.
    4. Salva il dataframe risultato dell'analisi in un nuovo dataframe.
    5. Elimina i record duplicati (tenendo traccia di quelli rimossi in 'log_analysis').
    6. Salva entrambi i dataframe in formato CSV.
    """
    try:
        # 1. Connessione al database
        conn = psycopg2.connect(
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            host=DB_HOST,
            port=DB_PORT
        )
        logging.info("Connessione al database '%s' avvenuta con successo.", DB_NAME)

        # 2. Lettura dati dalla tabella 'news_cyber' (schema 'cyber_risk')
        query = """
            SELECT *
            FROM cyber_risk.news_cyber
        """
        df_original = pd.read_sql_query(query, con=conn)
        logging.info("Lettura dati completata. Numero di record: %d", len(df_original))

    except psycopg2.Error as e:
        logging.error("Errore nella connessione o esecuzione della query: %s", e)
        return  # Interrompiamo l'esecuzione poiché non possiamo procedere senza dati
    finally:
        # Chiusura della connessione al database
        try:
            conn.close()
            logging.info("Connessione al database chiusa correttamente.")
        except Exception as e:
            logging.warning("Impossibile chiudere la connessione: %s", e)

    try:
        # 3. Applica le procedure di cleaning sul campo 'Body'
        df_processed = df_original.copy()
        df_processed['Body'] = df_processed['Body'].apply(clean_text)

        # 4. Salva il dataframe risultato dell'analisi in un nuovo dataframe
        df_cleaned = df_processed.copy()
        logging.info("Pulizia del testo completata.")

        # 5. Elimina i record duplicati e crea log dei record eliminati
        duplicates_mask = df_cleaned.duplicated(keep='first')
        log_analysis = df_cleaned[duplicates_mask].copy()  # record duplicati
        df_cleaned.drop_duplicates(keep='first', inplace=True)

        logging.info(
            "Rimozione dei record duplicati completata. "
            "Record totali rimanenti: %d, record duplicati: %d",
            len(df_cleaned), len(log_analysis)
        )

        # 6. Salva entrambi i dataframe in formato CSV
        df_cleaned.to_csv("df_cleaned.csv", index=False)
        log_analysis.to_csv("log_analysis.csv", index=False)
        logging.info("DataFrame pulito e log di duplicati salvati in formato CSV.")

    except KeyError as e:
        logging.error("Errore nell'accesso a una colonna del dataframe: %s", e)
    except Exception as e:
        logging.error("Errore imprevisto nella pulizia dei testi: %s", e)

if __name__ == "__main__":
    main()
```

---

### Spiegazione dei punti chiave

1. **Connessione a PostgreSQL**  
   Utilizziamo `psycopg2.connect()` in un blocco `try-except` per catturare eventuali errori di connessione o di query. Nel blocco `finally`, cerchiamo di chiudere la connessione anche in caso di errore.

2. **Lettura e conversione in DataFrame**  
   `pd.read_sql_query()` legge i dati direttamente in un DataFrame di pandas.  

3. **Funzione di pulizia `clean_text()`**  
   - **Conversione in minuscolo**: uniforma il testo.  
   - **Rimozione di punteggiatura**: utilizza `str.translate()` per eliminare i caratteri definiti in `string.punctuation`.  
   - **Rimozione di numeri**: con `re.sub(r'\d+', '', text)`.  
   - **Rimozione di caratteri speciali**: con `re.sub(r'[^a-z0-9\s]', ' ', text)`.  
   - **Tokenizzazione**: con `word_tokenize(text)`.  
   - **Rimozione stopwords (inglese)**: usando la lista `stopwords.words('english')` di nltk.  
   - **Riassemblaggio**: unisce i token filtrati in una stringa pulita.  

4. **Gestione duplicati**  
   - `df_cleaned.duplicated(keep='first')` individua i duplicati.  
   - `log_analysis` tiene traccia dei record duplicati prima della rimozione.  
   - `df_cleaned.drop_duplicates()` rimuove i duplicati.  

5. **Salvataggio finale**  
   - `df_cleaned.to_csv("df_cleaned.csv")` e `log_analysis.to_csv("log_analysis.csv")` per esportare in CSV.  

6. **Gestione degli errori**  
   - `except psycopg2.Error as e:` cattura gli errori legati al database.  
   - `except KeyError as e:` gestisce l’accesso a colonne che potrebbero non esistere.  
   - `except Exception as e:` cattura altri errori imprevisti.  

Questa implementazione garantisce un processo completo e robusto: dalla connessione al database all’esportazione dei risultati finali, con ampio uso di logging per tracciare ogni fase.

## Richiesta 3

modifica il codice scritto in modo da rimuovere non i record uguali bensì quelli che hanno lo stesso identico testo nel campo "body"

Di seguito trovi il codice aggiornato affinché vengano considerati duplicati **solo** i record che hanno lo stesso identico testo nel campo **"Body"**, ignorando gli altri campi. In altre parole, vengono rimossi i record che hanno un campo **"Body"** duplicato, mantenendo il primo e rimuovendo tutti gli altri.

```python
import psycopg2
import pandas as pd
import logging

try:
    import nltk
    nltk.download('punkt')
    nltk.download('stopwords')
    from nltk.corpus import stopwords
    from nltk.tokenize import word_tokenize
    import string
except ImportError as e:
    logging.error("Errore nell'importare la libreria di NLP: %s", e)
    raise  # Interrompiamo l'esecuzione se non possiamo effettuare il cleaning del testo

# Configurazione di base per il logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Parametri di connessione al database (modificare in base alla propria configurazione)
DB_NAME = "text_analysis"
DB_USER = "postgres"
DB_PASSWORD = "password"
DB_HOST = "localhost"
DB_PORT = "5432"


def clean_text(text: str) -> str:
    """
    Esegue la pulizia di base di un testo in lingua inglese:
    1. Conversione in minuscolo
    2. Rimozione della punteggiatura
    3. Rimozione dei numeri
    4. Rimozione di eventuali caratteri speciali
    5. Tokenizzazione
    6. Rimozione stopwords (lingua inglese)
    7. Riassemblaggio del testo

    Restituisce la stringa pulita.
    """
    if not isinstance(text, str):
        # Se 'text' non è una stringa, restituisce stringa vuota
        return ""

    # 1. Conversione in minuscolo
    text = text.lower()

    # 2. Rimozione della punteggiatura
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 3. Rimozione dei numeri
    import re
    text = re.sub(r'\d+', '', text)

    # 4. Rimozione di eventuali caratteri speciali aggiuntivi (se necessario)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # 5. Tokenizzazione
    tokens = word_tokenize(text)

    # 6. Rimozione stopwords (lingua inglese)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    # 7. Riassemblaggio del testo
    cleaned_text = " ".join(tokens)
    
    return cleaned_text


def main():
    """
    Funzione principale che:
    1. Legge tutti i dati dalla tabella 'news_cyber' dello schema 'cyber_risk'
       nel database 'text_analysis'.
    2. Trasferisce i dati in un dataframe pandas.
    3. Applica il cleaning sul campo 'Body'.
    4. Salva il dataframe risultato dell'analisi in un nuovo dataframe.
    5. Elimina i record duplicati *solo se hanno lo stesso testo in 'Body'*, 
       tenendo traccia di quelli rimossi in 'log_analysis'.
    6. Salva entrambi i dataframe in formato CSV.
    """
    try:
        # 1. Connessione al database
        conn = psycopg2.connect(
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            host=DB_HOST,
            port=DB_PORT
        )
        logging.info("Connessione al database '%s' avvenuta con successo.", DB_NAME)

        # 2. Lettura dati dalla tabella 'news_cyber' (schema 'cyber_risk')
        query = """
            SELECT *
            FROM cyber_risk.news_cyber
        """
        df_original = pd.read_sql_query(query, con=conn)
        logging.info("Lettura dati completata. Numero di record: %d", len(df_original))

    except psycopg2.Error as e:
        logging.error("Errore nella connessione o esecuzione della query: %s", e)
        return  # Interrompiamo l'esecuzione poiché non possiamo procedere senza dati
    finally:
        # Chiusura della connessione al database
        try:
            conn.close()
            logging.info("Connessione al database chiusa correttamente.")
        except Exception as e:
            logging.warning("Impossibile chiudere la connessione: %s", e)

    try:
        # 3. Applica le procedure di cleaning sul campo 'Body'
        df_processed = df_original.copy()
        df_processed['Body'] = df_processed['Body'].apply(clean_text)

        # 4. Salva il dataframe risultato dell'analisi in un nuovo dataframe
        df_cleaned = df_processed.copy()
        logging.info("Pulizia del testo completata.")

        # 5. Elimina i record duplicati *solo* in base al testo di 'Body'
        #    -> subset=['Body']
        duplicates_mask = df_cleaned.duplicated(subset=['Body'], keep='first')
        log_analysis = df_cleaned[duplicates_mask].copy()  # record duplicati
        df_cleaned.drop_duplicates(subset=['Body'], keep='first', inplace=True)

        logging.info(
            "Rimozione dei record duplicati (in base a 'Body') completata. "
            "Record totali rimanenti: %d, record duplicati: %d",
            len(df_cleaned), len(log_analysis)
        )

        # 6. Salva entrambi i dataframe in formato CSV
        df_cleaned.to_csv("df_cleaned.csv", index=False)
        log_analysis.to_csv("log_analysis.csv", index=False)
        logging.info("DataFrame pulito e log di duplicati salvati in formato CSV.")

    except KeyError as e:
        logging.error("Errore nell'accesso a una colonna del dataframe: %s", e)
    except Exception as e:
        logging.error("Errore imprevisto nella pulizia dei testi: %s", e)


if __name__ == "__main__":
    main()
```

---

## Spiegazione delle modifiche al passaggio 5

- Per **trovare i record duplicati** solo in base al testo del campo **"Body"** utilizziamo:
  ```python
  duplicates_mask = df_cleaned.duplicated(subset=['Body'], keep='first')
  ```
  In questo modo, `pandas` controllerà i duplicati **esclusivamente** sulla colonna `Body`.

- Per **rimuovere i duplicati**, usiamo:
  ```python
  df_cleaned.drop_duplicates(subset=['Body'], keep='first', inplace=True)
  ```
  Questa istruzione mantiene soltanto il **primo** record trovato per ogni valore di `Body` e rimuove tutti gli altri.

- **`log_analysis`** memorizza i record che risultano duplicati, ossia quelli contrassegnati da `duplicates_mask`. In questo modo teniamo traccia di quali righe sono state effettivamente rimosse.

## Richiesta 4

Nel dataframe df_cleaned rimuovi tutti i record che hanno il campo pub_data nullo o vuoto

Di seguito trovi il codice aggiornato con l’aggiunta di un passaggio per **rimuovere tutti i record** di **`df_cleaned`** che hanno il campo **`pub_data`** nullo oppure vuoto (stringa vuota). 

```python
import psycopg2
import pandas as pd
import logging

try:
    import nltk
    nltk.download('punkt')
    nltk.download('stopwords')
    from nltk.corpus import stopwords
    from nltk.tokenize import word_tokenize
    import string
except ImportError as e:
    logging.error("Errore nell'importare la libreria di NLP: %s", e)
    raise  # Interrompiamo l'esecuzione se non possiamo effettuare il cleaning del testo

# Configurazione di base per il logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Parametri di connessione al database (modificare in base alla propria configurazione)
DB_NAME = "text_analysis"
DB_USER = "postgres"
DB_PASSWORD = "password"
DB_HOST = "localhost"
DB_PORT = "5432"


def clean_text(text: str) -> str:
    """
    Esegue la pulizia di base di un testo in lingua inglese:
    1. Conversione in minuscolo
    2. Rimozione della punteggiatura
    3. Rimozione dei numeri
    4. Rimozione di eventuali caratteri speciali
    5. Tokenizzazione
    6. Rimozione stopwords (lingua inglese)
    7. Riassemblaggio del testo

    Restituisce la stringa pulita.
    """
    if not isinstance(text, str):
        # Se 'text' non è una stringa, restituisce stringa vuota
        return ""

    # 1. Conversione in minuscolo
    text = text.lower()

    # 2. Rimozione della punteggiatura
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 3. Rimozione dei numeri
    import re
    text = re.sub(r'\d+', '', text)

    # 4. Rimozione di eventuali caratteri speciali aggiuntivi (se necessario)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # 5. Tokenizzazione
    tokens = word_tokenize(text)

    # 6. Rimozione stopwords (lingua inglese)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    # 7. Riassemblaggio del testo
    cleaned_text = " ".join(tokens)
    
    return cleaned_text


def main():
    """
    Funzione principale che:
    1. Legge tutti i dati dalla tabella 'news_cyber' dello schema 'cyber_risk'
       nel database 'text_analysis'.
    2. Trasferisce i dati in un dataframe pandas.
    3. Applica il cleaning sul campo 'Body'.
    4. Salva il dataframe risultato dell'analisi in un nuovo dataframe.
    5. Elimina i record duplicati *solo se hanno lo stesso testo in 'Body'*, 
       tenendo traccia di quelli rimossi in 'log_analysis'.
    6. Rimuove dal dataframe finale tutti i record con 'pub_data' nullo o vuoto.
    7. Salva entrambi i dataframe in formato CSV.
    """
    try:
        # 1. Connessione al database
        conn = psycopg2.connect(
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            host=DB_HOST,
            port=DB_PORT
        )
        logging.info("Connessione al database '%s' avvenuta con successo.", DB_NAME)

        # 2. Lettura dati dalla tabella 'news_cyber' (schema 'cyber_risk')
        query = """
            SELECT *
            FROM cyber_risk.news_cyber
        """
        df_original = pd.read_sql_query(query, con=conn)
        logging.info("Lettura dati completata. Numero di record: %d", len(df_original))

    except psycopg2.Error as e:
        logging.error("Errore nella connessione o esecuzione della query: %s", e)
        return  # Interrompiamo l'esecuzione poiché non possiamo procedere senza dati
    finally:
        # Chiusura della connessione al database
        try:
            conn.close()
            logging.info("Connessione al database chiusa correttamente.")
        except Exception as e:
            logging.warning("Impossibile chiudere la connessione: %s", e)

    try:
        # 3. Applica le procedure di cleaning sul campo 'Body'
        df_processed = df_original.copy()
        df_processed['Body'] = df_processed['Body'].apply(clean_text)

        # 4. Salva il dataframe risultato dell'analisi in un nuovo dataframe
        df_cleaned = df_processed.copy()
        logging.info("Pulizia del testo completata.")

        # 5. Elimina i record duplicati *solo* in base al testo di 'Body'
        duplicates_mask = df_cleaned.duplicated(subset=['Body'], keep='first')
        log_analysis = df_cleaned[duplicates_mask].copy()  # record duplicati
        df_cleaned.drop_duplicates(subset=['Body'], keep='first', inplace=True)
        logging.info(
            "Rimozione dei record duplicati (in base a 'Body') completata. "
            "Record totali rimanenti: %d, record duplicati: %d",
            len(df_cleaned), len(log_analysis)
        )

        # 6. Rimuove i record che hanno 'pub_data' nullo o vuoto
        #    - Per nullo (NaN), usiamo notnull()
        #    - Per vuoto, controlliamo che non sia una stringa vuota dopo lo strip
        #    Se pub_data non è una stringa, convertila a string per sicurezza
        df_cleaned = df_cleaned[df_cleaned['pub_data'].notnull()]  # rimuove i NaN
        df_cleaned = df_cleaned[df_cleaned['pub_data'].astype(str).str.strip() != '']
        logging.info("Rimossi i record con 'pub_data' nullo o vuoto. Record totali rimanenti: %d", len(df_cleaned))

        # 7. Salva entrambi i dataframe in formato CSV
        df_cleaned.to_csv("df_cleaned.csv", index=False)
        log_analysis.to_csv("log_analysis.csv", index=False)
        logging.info("DataFrame pulito e log di duplicati salvati in formato CSV.")

    except KeyError as e:
        logging.error("Errore nell'accesso a una colonna del dataframe: %s", e)
    except Exception as e:
        logging.error("Errore imprevisto nella pulizia dei testi: %s", e)


if __name__ == "__main__":
    main()
```

### Cosa è cambiato?

1. **Rimozione record con `pub_data` nullo (NaN)**  
   ```python
   df_cleaned = df_cleaned[df_cleaned['pub_data'].notnull()]
   ```
   In questo modo eliminiamo tutti i record in cui il valore di `pub_data` è `NaN`.

2. **Rimozione record con `pub_data` vuoto**  
   ```python
   df_cleaned = df_cleaned[df_cleaned['pub_data'].astype(str).str.strip() != '']
   ```
   - Convertiamo `pub_data` in stringa per assicurarci di poter usare `strip()`.  
   - Con `str.strip()` rimuoviamo spazi bianchi iniziali e finali.  
   - Infine, controlliamo che il risultato non sia una stringa vuota. Se è vuota, il record viene rimosso.

3. **Logging**  
   - Aggiunto un messaggio per indicare quanti record sono rimasti dopo la rimozione di quelli con `pub_data` nullo o vuoto.

## Richiesta 5

add a procedure to save df_cleaned on a same db in a table denominated news_cyber_clean

Di seguito un esempio di codice aggiornato che, oltre alle operazioni richieste in precedenza, **salva** il DataFrame **`df_cleaned`** in **una nuova tabella** del **medesimo database** denominata **`news_cyber_clean`** all’interno dello **schema** `cyber_risk`.  
Per farlo, utilizziamo **SQLAlchemy** (in particolare la funzione `to_sql` di pandas) che semplifica l’inserimento di un DataFrame in un database SQL.  

Assicurati di aver installato il pacchetto SQLAlchemy se non presente:
```bash
pip install sqlalchemy
```

---

```python
import psycopg2
import pandas as pd
import logging

# Import per il salvataggio su DB tramite SQLAlchemy
from sqlalchemy import create_engine

try:
    import nltk
    nltk.download('punkt')
    nltk.download('stopwords')
    from nltk.corpus import stopwords
    from nltk.tokenize import word_tokenize
    import string
except ImportError as e:
    logging.error("Errore nell'importare la libreria di NLP: %s", e)
    raise  # Interrompiamo l'esecuzione se non possiamo effettuare il cleaning del testo

# Configurazione di base per il logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Parametri di connessione al database (modificare in base alla propria configurazione)
DB_NAME = "text_analysis"
DB_USER = "postgres"
DB_PASSWORD = "password"
DB_HOST = "localhost"
DB_PORT = "5432"

def clean_text(text: str) -> str:
    """
    Esegue la pulizia di base di un testo in lingua inglese:
    1. Conversione in minuscolo
    2. Rimozione della punteggiatura
    3. Rimozione dei numeri
    4. Rimozione di eventuali caratteri speciali
    5. Tokenizzazione
    6. Rimozione stopwords (lingua inglese)
    7. Riassemblaggio del testo

    Restituisce la stringa pulita.
    """
    if not isinstance(text, str):
        # Se 'text' non è una stringa, restituisce stringa vuota
        return ""

    # 1. Conversione in minuscolo
    text = text.lower()

    # 2. Rimozione della punteggiatura
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 3. Rimozione dei numeri
    import re
    text = re.sub(r'\d+', '', text)

    # 4. Rimozione di eventuali caratteri speciali
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # 5. Tokenizzazione
    tokens = word_tokenize(text)

    # 6. Rimozione stopwords (lingua inglese)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    # 7. Riassemblaggio del testo
    cleaned_text = " ".join(tokens)

    return cleaned_text


def main():
    """
    Funzione principale che:
    1. Legge tutti i dati dalla tabella 'news_cyber' dello schema 'cyber_risk'
       nel database 'text_analysis'.
    2. Trasferisce i dati in un dataframe pandas.
    3. Applica il cleaning sul campo 'Body'.
    4. Salva il dataframe risultato dell'analisi in un nuovo dataframe (df_cleaned).
    5. Elimina i record duplicati (in base a 'Body'), tenendo traccia di quelli rimossi in 'log_analysis'.
    6. Rimuove dal dataframe finale i record con 'pub_data' nullo o vuoto.
    7. Salva df_cleaned e log_analysis in formato CSV.
    8. Salva df_cleaned in una tabella 'news_cyber_clean' nello schema 'cyber_risk'.
    """
    try:
        # 1. Connessione al database con psycopg2
        conn = psycopg2.connect(
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            host=DB_HOST,
            port=DB_PORT
        )
        logging.info("Connessione al database '%s' avvenuta con successo.", DB_NAME)

        # 2. Lettura dati dalla tabella 'news_cyber' (schema 'cyber_risk')
        query = """
            SELECT *
            FROM cyber_risk.news_cyber
        """
        df_original = pd.read_sql_query(query, con=conn)
        logging.info("Lettura dati completata. Numero di record: %d", len(df_original))

    except psycopg2.Error as e:
        logging.error("Errore nella connessione o esecuzione della query: %s", e)
        return  # Interrompiamo l'esecuzione poiché non possiamo procedere senza dati
    finally:
        # Chiusura della connessione al database
        try:
            conn.close()
            logging.info("Connessione al database chiusa correttamente.")
        except Exception as e:
            logging.warning("Impossibile chiudere la connessione: %s", e)

    try:
        # 3. Applica le procedure di cleaning sul campo 'Body'
        df_processed = df_original.copy()
        df_processed['Body'] = df_processed['Body'].apply(clean_text)

        # 4. Salva il dataframe risultato dell'analisi in un nuovo dataframe
        df_cleaned = df_processed.copy()
        logging.info("Pulizia del testo completata.")

        # 5. Elimina i record duplicati *solo* in base al testo di 'Body'
        duplicates_mask = df_cleaned.duplicated(subset=['Body'], keep='first')
        log_analysis = df_cleaned[duplicates_mask].copy()  # record duplicati
        df_cleaned.drop_duplicates(subset=['Body'], keep='first', inplace=True)

        logging.info(
            "Rimozione dei record duplicati (in base a 'Body') completata. "
            "Record totali rimanenti: %d, record duplicati: %d",
            len(df_cleaned), len(log_analysis)
        )

        # 6. Rimuove i record che hanno 'pub_data' nullo o vuoto
        df_cleaned = df_cleaned[df_cleaned['pub_data'].notnull()]  # rimuove i NaN
        df_cleaned = df_cleaned[df_cleaned['pub_data'].astype(str).str.strip() != '']
        logging.info("Rimossi i record con 'pub_data' nullo o vuoto. Record totali rimanenti: %d", len(df_cleaned))

        # 7. Salva entrambi i dataframe in formato CSV
        df_cleaned.to_csv("df_cleaned.csv", index=False)
        log_analysis.to_csv("log_analysis.csv", index=False)
        logging.info("DataFrame pulito e log di duplicati salvati in formato CSV.")

        # 8. Salva df_cleaned in una nuova tabella 'news_cyber_clean' (schema 'cyber_risk') usando SQLAlchemy
        #    Creiamo un engine SQLAlchemy per la connessione
        try:
            # Crea la stringa di connessione con i parametri sopra
            engine_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
            engine = create_engine(engine_url)

            # Salvataggio su tabella 'news_cyber_clean' nello schema 'cyber_risk'
            df_cleaned.to_sql(
                name='news_cyber_clean',     # nome della tabella
                con=engine,
                schema='cyber_risk',        # schema di destinazione
                if_exists='replace',        # sovrascrive la tabella se esiste
                index=False
            )
            logging.info("DataFrame salvato con successo nella tabella 'news_cyber_clean' (schema 'cyber_risk').")

        except Exception as e:
            logging.error("Errore durante il salvataggio del DataFrame in DB: %s", e)

        finally:
            # Chiudiamo il motore SQLAlchemy
            try:
                engine.dispose()
                logging.info("Connessione con SQLAlchemy chiusa correttamente.")
            except Exception as e:
                logging.warning("Impossibile chiudere l'engine SQLAlchemy: %s", e)

    except KeyError as e:
        logging.error("Errore nell'accesso a una colonna del dataframe: %s", e)
    except Exception as e:
        logging.error("Errore imprevisto nella pulizia dei testi o nel salvataggio: %s", e)


if __name__ == "__main__":
    main()
```

---

## Spiegazione della nuova parte di codice (punto 8)

1. **Creazione Engine SQLAlchemy**  
   ```python
   engine_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
   engine = create_engine(engine_url)
   ```
   - Costruiamo una stringa di connessione con i parametri del database.  
   - Creiamo un `engine` che ci permette di scrivere (o leggere) da/verso il DB.

2. **Scrittura del DataFrame**  
   ```python
   df_cleaned.to_sql(
       name='news_cyber_clean',
       con=engine,
       schema='cyber_risk',
       if_exists='replace',
       index=False
   )
   ```
   - **`name='news_cyber_clean'`** specifica il nome della tabella di destinazione.  
   - **`schema='cyber_risk'`** indica lo schema in cui salvare la tabella.  
   - **`if_exists='replace'`** indica di sostituire la tabella se già esiste. Se vuoi aggiungere dati invece che sovrascrivere, puoi usare `if_exists='append'`.  
   - **`index=False`** evita di scrivere l’indice di pandas come colonna.

3. **Chiusura dell’engine**  
   - Per pulizia, effettuiamo un `engine.dispose()` in un blocco `finally`, così da liberare le risorse anche in caso di errore.

In questo modo, alla fine dell’esecuzione, avremo:

- I file **`df_cleaned.csv`** e **`log_analysis.csv`** nella cartella di lavoro.  
- Una nuova tabella denominata **`news_cyber_clean`** (nello **schema `cyber_risk`**) nel database **`text_analysis`**, popolata con i dati ripuliti di **`df_cleaned`**.

## Richiesta 6

la nuova procedura produce il seguente errore: Errore durante il salvataggio del DataFrame in DB: name 'create_engine' is not defined

Il messaggio di errore:

```
Errore durante il salvataggio del DataFrame in DB: name 'create_engine' is not defined
```

indica che la funzione **`create_engine`** non viene riconosciuta all’interno del codice. Questo accade tipicamente per uno dei seguenti motivi:

1. **Manca l’import di `create_engine`** da SQLAlchemy.
2. **SQLAlchemy non è installato** nell’ambiente Python in cui stai eseguendo il codice.
3. Stai eseguendo il codice in un **ambiente diverso** (ad esempio un virtual environment) da quello in cui hai installato SQLAlchemy.
4. C’è stato un **errore di copiaincolla** del codice, e la riga di import è andata persa.

---

## Come risolvere

1. **Verifica di aver importato correttamente la funzione**:
   ```python
   from sqlalchemy import create_engine
   ```
   Se questa riga non è presente, aggiungila all’inizio del tuo script, insieme agli altri import.

2. **Installa SQLAlchemy** (se necessario):
   ```bash
   pip install sqlalchemy
   ```
   Assicurati di usare lo stesso interprete Python (o la stessa virtual environment) in cui esegui lo script.

3. **Controlla l’ambiente Python**:  
   Se utilizzi virtual environment (ad esempio con `venv` o `conda`), verifica di aver attivato quello corretto:
   ```bash
   # Esempio se usi venv
   source /path/to/venv/bin/activate
   pip install sqlalchemy
   python script.py
   ```

---

### Esempio di codice funzionante

Ecco un esempio di codice **completo** che include l’import di `create_engine` e che dovrebbe risolvere il problema:

```python
import psycopg2
import pandas as pd
import logging

# Import per il salvataggio su DB tramite SQLAlchemy
from sqlalchemy import create_engine

try:
    import nltk
    nltk.download('punkt')
    nltk.download('stopwords')
    from nltk.corpus import stopwords
    from nltk.tokenize import word_tokenize
    import string
except ImportError as e:
    logging.error("Errore nell'importare la libreria di NLP: %s", e)
    raise  # Interrompiamo l'esecuzione se non possiamo effettuare il cleaning del testo

# Configurazione di base per il logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Parametri di connessione al database (modificare in base alla propria configurazione)
DB_NAME = "text_analysis"
DB_USER = "postgres"
DB_PASSWORD = "password"
DB_HOST = "localhost"
DB_PORT = "5432"

def clean_text(text: str) -> str:
    """
    Esegue la pulizia di base di un testo in lingua inglese:
    1. Conversione in minuscolo
    2. Rimozione della punteggiatura
    3. Rimozione dei numeri
    4. Rimozione di eventuali caratteri speciali
    5. Tokenizzazione
    6. Rimozione stopwords (lingua inglese)
    7. Riassemblaggio del testo

    Restituisce la stringa pulita.
    """
    if not isinstance(text, str):
        # Se 'text' non è una stringa, restituisce stringa vuota
        return ""

    # 1. Conversione in minuscolo
    text = text.lower()

    # 2. Rimozione della punteggiatura
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 3. Rimozione dei numeri
    import re
    text = re.sub(r'\d+', '', text)

    # 4. Rimozione di eventuali caratteri speciali
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # 5. Tokenizzazione
    tokens = word_tokenize(text)

    # 6. Rimozione stopwords (lingua inglese)
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    # 7. Riassemblaggio del testo
    cleaned_text = " ".join(tokens)

    return cleaned_text


def main():
    """
    Funzione principale che:
    1. Legge tutti i dati dalla tabella 'news_cyber' dello schema 'cyber_risk'
       nel database 'text_analysis'.
    2. Trasferisce i dati in un dataframe pandas.
    3. Applica il cleaning sul campo 'Body'.
    4. Elimina i record duplicati in base al campo 'Body' e crea un log di quelli rimossi.
    5. Rimuove i record con 'pub_data' nullo o vuoto.
    6. Salva df_cleaned e log_analysis in formato CSV.
    7. Salva df_cleaned in una tabella 'news_cyber_clean' nello schema 'cyber_risk'.
    """
    try:
        # 1. Connessione al database con psycopg2
        conn = psycopg2.connect(
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            host=DB_HOST,
            port=DB_PORT
        )
        logging.info("Connessione al database '%s' avvenuta con successo.", DB_NAME)

        # 2. Lettura dati dalla tabella 'news_cyber' (schema 'cyber_risk')
        query = """
            SELECT *
            FROM cyber_risk.news_cyber
        """
        df_original = pd.read_sql_query(query, con=conn)
        logging.info("Lettura dati completata. Numero di record: %d", len(df_original))

    except psycopg2.Error as e:
        logging.error("Errore nella connessione o esecuzione della query: %s", e)
        return  # Interrompiamo l'esecuzione poiché non possiamo procedere senza dati
    finally:
        # Chiusura della connessione al database
        try:
            conn.close()
            logging.info("Connessione al database chiusa correttamente.")
        except Exception as e:
            logging.warning("Impossibile chiudere la connessione: %s", e)

    try:
        # 3. Pulizia del campo 'Body'
        df_processed = df_original.copy()
        df_processed['Body'] = df_processed['Body'].apply(clean_text)

        # Copio in un nuovo dataframe
        df_cleaned = df_processed.copy()
        logging.info("Pulizia del testo completata.")

        # 4. Duplicati in base a 'Body'
        duplicates_mask = df_cleaned.duplicated(subset=['Body'], keep='first')
        log_analysis = df_cleaned[duplicates_mask].copy()
        df_cleaned.drop_duplicates(subset=['Body'], keep='first', inplace=True)
        logging.info(
            "Rimozione dei record duplicati (in base a 'Body') completata. "
            "Record totali rimanenti: %d, record duplicati: %d",
            len(df_cleaned), len(log_analysis)
        )

        # 5. Rimuovere i record con 'pub_data' nullo o vuoto
        df_cleaned = df_cleaned[df_cleaned['pub_data'].notnull()]
        df_cleaned = df_cleaned[df_cleaned['pub_data'].astype(str).str.strip() != '']
        logging.info("Rimossi i record con 'pub_data' nullo o vuoto. Record totali rimanenti: %d", len(df_cleaned))

        # 6. Salvataggio in CSV
        df_cleaned.to_csv("df_cleaned.csv", index=False)
        log_analysis.to_csv("log_analysis.csv", index=False)
        logging.info("DataFrame pulito e log di duplicati salvati in formato CSV.")

        # 7. Salvataggio su DB con SQLAlchemy
        try:
            engine_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
            engine = create_engine(engine_url)

            df_cleaned.to_sql(
                name='news_cyber_clean',  # tabella di destinazione
                con=engine,
                schema='cyber_risk',     # schema
                if_exists='replace',     # sostituisce la tabella se esiste
                index=False
            )
            logging.info("DataFrame salvato con successo nella tabella 'news_cyber_clean' (schema 'cyber_risk').")

        except Exception as e:
            logging.error("Errore durante il salvataggio del DataFrame in DB: %s", e)

        finally:
            try:
                engine.dispose()
                logging.info("Connessione con SQLAlchemy chiusa correttamente.")
            except Exception as e:
                logging.warning("Impossibile chiudere l'engine SQLAlchemy: %s", e)

    except KeyError as e:
        logging.error("Errore nell'accesso a una colonna del dataframe: %s", e)
    except Exception as e:
        logging.error("Errore imprevisto nella pulizia dei testi o nel salvataggio: %s", e)


if __name__ == "__main__":
    main()
```

**Ricorda**: se ancora ottieni l’errore, è molto probabile che **SQLAlchemy non sia installato** nel tuo ambiente Python corrente. In tal caso, verifica l’installazione e l’ambiente di lavoro.

## Richiesta 7

con riferimento alla tabelle news_cyber_clean appena creata, scrivi una procedura sql funzionante per postgressql che costruisca un'altra tabella denominata news_cyber_daily che contenga due colonne: 1) la prima denominata "date" deve contenere tutte le date dal 1/1/2014 al 31/12/2022, 2) la seconda deve contenere la somma del numero di record che, nella tabella news_cyber_clean hanno data corrispondente alla data considerata. Qualora per una particolare data non si trovino record corrispondenti introdurre 0

Di seguito trovi un esempio di **istruzione SQL** funzionante in **PostgreSQL** che crea la tabella **`news_cyber_daily`** (all’interno dello *schema* `cyber_risk`) contenente:

1. Una colonna `date` con **tutte le date dal 1° gennaio 2014 al 31 dicembre 2022**.  
2. Una colonna (in questo esempio la chiamiamo `total_records`) che indica **il numero di record** con la stessa data (campo `pub_data`) nella tabella `news_cyber_clean`. Se nessun record corrisponde a una determinata data, verrà inserito il valore **0**.

> **Assunzioni**:
> - La colonna `pub_data` in `news_cyber_clean` è di tipo `date`, o comunque convertibile a `date`.  
> - Se `pub_data` non è di tipo `date` (ad esempio `timestamp` o `text`), occorrerà castarla (es. `pub_data::date`) sia in `SELECT` sia in `JOIN`.  
> - La nuova tabella `news_cyber_daily` viene creata con la sintassi `CREATE TABLE ... AS ...`. Se la tabella esiste già, PostgreSQL restituirà errore (a meno di gestione esplicita con `DROP TABLE` o `CREATE OR REPLACE TABLE`).  
> - Se vuoi che la tabella venga creata in un altro schema (ad esempio `public`), sostituisci il nome dello schema nel comando SQL.

---

## Esempio di istruzione SQL

```sql
CREATE TABLE cyber_risk.news_cyber_daily AS
SELECT 
    d.dt AS date,
    COALESCE(t.cnt, 0) AS total_records
FROM generate_series(
        '2014-01-01'::date, 
        '2022-12-31'::date, 
        interval '1 day'
     ) AS d(dt)
LEFT JOIN (
    SELECT 
        pub_data AS this_date, 
        COUNT(*) AS cnt
    FROM cyber_risk.news_cyber_clean
    GROUP BY pub_data
) AS t
ON t.this_date = d.dt
ORDER BY d.dt;
```

### Spiegazione

1. **generate_series('2014-01-01'::date, '2022-12-31'::date, interval '1 day')**  
   - Genera tutte le date giornaliere comprese tra il 1° gennaio 2014 e il 31 dicembre 2022.  
   - L’alias `d(dt)` fa sì che ogni riga abbia un campo `dt` che rappresenta la data del giorno.

2. **Sotto-query con GROUP BY**  
   ```sql
   SELECT 
       pub_data AS this_date, 
       COUNT(*) AS cnt
   FROM cyber_risk.news_cyber_clean
   GROUP BY pub_data
   ```
   - Estrae dalla tabella `news_cyber_clean` il numero totale di record per ogni data (`pub_data`).  
   - L’alias `this_date` identifica la colonna data in questa sotto-query, mentre `cnt` rappresenta il conteggio dei record.

3. **LEFT JOIN**  
   ```sql
   LEFT JOIN (
       ...
   ) AS t
   ON t.this_date = d.dt
   ```
   - Facciamo un `LEFT JOIN` tra la generazione di date (tutte le date) e la sotto-query che conta i record per data.  
   - In questo modo, se una determinata data non esiste in `news_cyber_clean`, la parte destra (`t.this_date`) sarà `NULL`; e grazie alla funzione `COALESCE(...)`, assegniamo **0** a `total_records`.

4. **COALESCE(t.cnt, 0)**  
   - Se non ci sono record per una determinata data, `t.cnt` risulterà `NULL`.  
   - `COALESCE(..., 0)` converte `NULL` in 0.

5. **CREATE TABLE ... AS**  
   - Crea effettivamente la tabella `news_cyber_daily` utilizzando i dati risultanti dalla query.  
   - Avrà due colonne: 
     - `date` (tipo *date*)  
     - `total_records` (tipo *bigint* o *int*, a seconda di come PostgreSQL gestisce il `COUNT(*)`).

6. **ORDER BY d.dt**  
   - Ordina i risultati finali in ordine cronologico, dal 1° gennaio 2014 al 31 dicembre 2022.

---

### Caso particolare: `pub_data` non è di tipo `date`

Se la colonna `pub_data` in `news_cyber_clean` non è di tipo `date` (ad esempio `timestamp`), sostituisci sia nel `SELECT` interno che nella condizione di join la colonna con il cast a date, ad esempio:

```sql
SELECT 
    pub_data::date AS this_date,
    COUNT(*) AS cnt
FROM cyber_risk.news_cyber_clean
GROUP BY pub_data::date
```

e

```sql
ON t.this_date = d.dt
```
resta invariato poiché `t.this_date` ora è di tipo date.